# RAG01｜建立 iPAS AI 應用規劃師學習知識庫

我想做一個協助複習「AI 應用規劃師中級科目一」的問答工具。這份程式先把官方學習指引變成可搜尋的知識庫；下一份 RAG02 才根據搜尋結果回答。

**操作方式：使用 Google Colab，由上往下一格一格執行；先讀說明，再看程式與輸出。** 建議選 GPU，CPU 也可執行但較慢。首次下載模型需要網路。

流程：官方 PDF → 排除前置頁與章末題解 → 擷取正文 → 切成短段落 → EmbeddingGemma 2 向量 → FAISS → 匯出 ZIP → 手動上傳 GitHub。

沿用老師的 `extract_rag_text()`、`extract_metadata()`、同模型版本、向量正規化、FAISS 精確搜尋與四檔匯出設計。原範例是「短文一篇一向量」；這份 PDF 較長，改成「頁內切段，一段一向量」。每段保留 PDF 頁碼與書內頁碼。

本份不呼叫 LLM，不需要 Groq 金鑰，也不會訓練模型。原始教材中的文字一律是資料，不能變成程式或系統指令。

老師的原始 Notebook 保留不動；這是另外製作的作業版本。官方指引是學習參考，不保證涵蓋實際考試全部內容。

## 1. 安裝套件：準備讀 PDF、做向量和搜尋的工具

沿用老師固定的 embedding 與 FAISS 版本，另加 `pypdf` 讀 PDF、`requests` 下載檔案。`pandas` 只用來讓檢查表容易閱讀。

**這一步完成後應沒有紅色錯誤。** 若本次工作階段已載入其他版本，安裝後請重新啟動工作階段，再從下一格繼續。不要一邊載入舊套件、一邊使用新版本。

模型的載入工具也會檢查 torchvision／torchaudio 依賴，即使只載入文字編碼器仍需備妥；Colab 通常已附帶，這裡一併確認安裝。

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1" "pypdf==6.12.0" "requests==2.32.5" "pandas>=2.2,<4" "torchvision" "torchaudio"

## 2. 集中設定：先決定資料、範圍與切段大小

這些設定放在一起，之後不用到各格尋找。`CHUNK_SIZE=800` 表示每段正文最多 800 個字元；`CHUNK_OVERLAP=120` 表示相鄰兩段重複 120 個字元，降低句子在邊界被切斷的影響。字元數不是模型的 token 數，後面還會檢查實際 token 長度。

`CONTENT_RANGES` 來自人工核對這個版本的 PDF：保留第三至第五章正文。PDF 頁碼從封面算第 1 頁；書內頁碼例如 `3-29` 是另一套編號，不能混用。來源檔案雜湊用來確認版本，檔案若改版就停止，不能沿用舊頁碼範圍。

GPU 支援 bfloat16 時使用它；其餘使用 float32，沿用老師避免 float16 的設定。

In [1]:
import numpy as np
import pandas as pd

In [2]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.metadata as package_metadata
import hashlib
import io
import json
import re
import tempfile
import zipfile

import requests
import torch
import faiss
from pypdf import PdfReader
from IPython.display import display
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

In [3]:
MODEL_ID = "google/embeddinggemma-2"
BATCH_SIZE = 4
EXPORT_ZIP_NAME = "ipas_vector_db.zip"
SOURCE_URL = 'https://www.ipas.org.tw/api/proxy/uploads/certification_resource/bf93f438f7be48d295c1b40a34d79f3d/AI%E6%87%89%E7%94%A8%E8%A6%8F%E5%8A%83%E5%B8%AB%28%E4%B8%AD%E7%B4%9A%29-%E5%AD%B8%E7%BF%92%E6%8C%87%E5%BC%95-%E7%A7%91%E7%9B%AE1%E4%BA%BA%E5%B7%A5%E6%99%BA%E6%85%A7%E6%8A%80%E8%A1%93%E6%87%89%E7%94%A8%E8%A6%8F%E5%8A%83_20251222101833.pdf'
SOURCE_FILENAME = "ipas_ai_intermediate_subject1.pdf"
EXPECTED_PDF_SHA256 = "7b1c67e3280600fbeca893bc1c13421d5aa44841272c34b456f1346731bf3812"
EXPECTED_PDF_PAGES = 168
CONTENT_RANGES = [(8, 90), (95, 137), (142, 163)]
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
DOCUMENT_TEMPLATE = "title: none | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"

# 起訖頁、所屬章節、書內頁碼的換算差。全部針對上面的 PDF 版本。
SECTION_RANGES = [
    (8, 9, "第三章 AI 相關技術應用｜概述", 7),
    (10, 46, "3.1 自然語言處理技術與應用", 7),
    (47, 65, "3.2 電腦視覺技術與應用", 7),
    (66, 79, "3.3 生成式 AI 技術與應用", 7),
    (80, 90, "3.4 多模態人工智慧應用", 7),
    (95, 96, "第四章 AI 導入評估規劃｜概述", 94),
    (97, 108, "4.1 AI 導入評估", 94),
    (109, 122, "4.2 AI 導入規劃", 94),
    (123, 137, "4.3 AI 風險管理", 94),
    (142, 142, "第五章 AI 技術應用與系統部署｜概述", 141),
    (143, 154, "5.1 數據準備與模型選擇", 141),
    (155, 163, "5.2 AI 技術系統集成與部署", 141),
]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
PACKAGE_NAMES = ["torch", "numpy", "sentence-transformers", "transformers", "faiss-cpu", "huggingface-hub", "pypdf", "requests", "pandas"]
PACKAGE_VERSIONS = {name: package_metadata.version(name) for name in PACKAGE_NAMES}
print("裝置：", DEVICE, "／模型精度：", DTYPE)
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["套件", "版本"]))

裝置： cpu ／模型精度： torch.float32


,套件,版本
0,torch,2.15.0
1,numpy,2.5.4
2,sentence-transformers,6.1.0
3,transformers,5.20.1
4,faiss-cpu,1.15.1
5,huggingface-hub,1.33.0
6,pypdf,6.12.0
7,requests,2.32.5
8,pandas,3.1.0


## 3. 下載指定版本的官方 PDF

程式從官方網址下載，不需要先手動轉成 TXT。先確認 HTTP 下載成功、內容真的以 PDF 標記開頭，再核對 SHA-256 與總頁數。SHA-256 可以想成檔案指紋：內容改一點，指紋就會不同。

如果官網暫時無法下載，可把 `PDF_INPUT_MODE` 改為 `"upload"`，上傳**同一版本**的 PDF。若指紋不同，請先核對新版章節與題解頁碼，再更新設定；不要只刪掉檢查。

In [4]:
PDF_INPUT_MODE = "download"  # 官網暫時無法存取時才改成 "upload"。
DATA_DIR = Path(tempfile.mkdtemp(prefix="ipas_pdf_"))
PDF_PATH = DATA_DIR / SOURCE_FILENAME
if PDF_INPUT_MODE == "download":
    response = requests.get(SOURCE_URL, timeout=(15, 90))
    response.raise_for_status()
    pdf_bytes = response.content
elif PDF_INPUT_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    names = [name for name in uploaded if name.lower().endswith(".pdf")]
    if len(names) != 1:
        raise ValueError("請一次上傳一個指定版本的 PDF。")
    pdf_bytes = uploaded[names[0]]
else:
    raise ValueError("PDF_INPUT_MODE 只能是 download 或 upload。")
if not pdf_bytes.startswith(b"%PDF-"):
    raise ValueError("下載內容不是 PDF，可能是官網錯誤頁。")
SOURCE_PDF_SHA256 = hashlib.sha256(pdf_bytes).hexdigest()
if SOURCE_PDF_SHA256 != EXPECTED_PDF_SHA256:
    raise ValueError("PDF 版本不同；請先核對頁碼範圍，再更新設定。")
PDF_PATH.write_bytes(pdf_bytes)
reader = PdfReader(io.BytesIO(pdf_bytes))
if len(reader.pages) != EXPECTED_PDF_PAGES:
    raise ValueError("PDF 頁數不同，請先核對資料版本。")
print("PDF 總頁數：", len(reader.pages))
print("檔案大小：", f"{len(pdf_bytes):,} bytes")
print("PDF SHA-256：", SOURCE_PDF_SHA256)

PDF 總頁數： 168
檔案大小： 4,541,416 bytes
PDF SHA-256： 7b1c67e3280600fbeca893bc1c13421d5aa44841272c34b456f1346731bf3812


## 4. 擷取正文頁，先看原始文字

只保留 PDF 第 8–90、95–137、142–163 頁，共 148 頁。第 1–7 頁是封面、序、目錄與考試介紹；第 91–94、138–141、164–167 頁是章末題目與解答；最後一頁不納入。

採用 `pypdf` 的一般文字模式。抽查時發現版面模式會打亂本 PDF 的部分英文字母順序，因此不用版面模式。這份作業只讀文字層，沒有辨識圖像或還原複雜表格；圖表相關問題仍須開原 PDF 核對。

先檢查一頁原始文字，再進行清理。換其他 PDF 時，這裡的頁碼範圍與章節設定都要重做。

In [5]:
selected_pages = [
    number for start, end in CONTENT_RANGES for number in range(start, end + 1)
]
if len(selected_pages) != len(set(selected_pages)):
    raise ValueError("正文頁碼範圍重複。")
excluded_pages = set(range(1, len(reader.pages) + 1)) - set(selected_pages)
assert len(selected_pages) == 148
assert set(range(91, 95)) | set(range(138, 142)) | set(range(164, 168)) <= excluded_pages
raw_pages = {number: reader.pages[number - 1].extract_text() or "" for number in selected_pages}
empty_pages = [number for number, text in raw_pages.items() if not text.strip()]
if empty_pages:
    raise ValueError(f"以下正文頁沒有可讀文字，需檢查或 OCR：{empty_pages}")
SAMPLE_PDF_PAGE = 36  # 這頁有 RAG 的說明；也可改成其他保留的頁碼。
sample_raw = raw_pages[SAMPLE_PDF_PAGE]
print("保留正文頁數：", len(raw_pages), "／排除頁數：", len(excluded_pages))
print("抽查 PDF 第", SAMPLE_PDF_PAGE, "頁原始文字：")
print(sample_raw[:1800])

保留正文頁數： 148 ／排除頁數： 20
抽查 PDF 第 36 頁原始文字：
 第三章 AI 相關技術應用 
3-29 
 檢索增強生成（ RAG） ：這一方法將「外部知識檢索」 與「語言生成」結
合，能夠在生成過程中整合大量外部資料，提升模型在背景知識要求較高
任務中的準確性。  
○ 示 例 ： 在 一 個 問 答 系 統 中 ，RAG 可 以 先 檢 索 出 相 關 的 資 料 （ 如 從
Wikipedia 檢索某一領域的資料） ， 再基於這些資料生成對應的答案 。 例
如，在解答「 What are the symptoms of COVID -19?」時， RAG 系統首
先會從公共知識庫檢索相關資料 ， 再生成更具體的答案 。 或是在醫療摘
要任務中， RAG 可以從醫學文獻資料庫中檢索最新的研究資料，並將
其結合生成診斷或治療建議。  
 低秩自適應（ LoRA）：LoRA 是一種使機器學習模型能夠快速適應新環境
的技術 ， 透過向原始模型添加輕量級的參數 ， 而非直接修改整體模型結構 ，
來幫助模型在特定應用中進行快速調整。  
這種方法使得大型和複雜的模型可以更靈活地適應不同的使用需求 ， 並能
在不大量重新訓練的情況下，高效地擴 充到更多情境。 LoRA 為開發人員
提供了一種有效的方式 ， 使他們能夠快速地調整和擴 充他們構建的機器學
習模型。  
○ 示例：假設我們要在資源有限的邊緣設備上部署大型預訓練模型（如
GPT-3） ，我們可以利用LoRA 技術來微調模型，僅調整必要的低秩參
數，從而大幅減少計算和 儲存需求，實現高效的部署。  
隨著預訓練語言模型的進步，結合語境理解、生成能力、外部知識強化和高
效微調策略等技術，讓現代 NLP 模型在實務應用中能夠更加靈活和高效。透過
RAG 和 LoRA 等新技術的應用，未來的預訓練模型將在多 工、多語言情境中發揮
更大的潛力，並且能夠快速適應不同語境和知識需求，提升各種自然語言處理任
務的表現。  
 
  


## 5. 第一個修改函式：取出適合 RAG 的文字

老師提醒「哪些文字要搜尋，也要交給 LLM」，這裡沿用同一原則。`extract_rag_text()` 移除重複頁首、單獨出現的書內頁碼及多餘空白，保留正文的段落、列表與專有名詞。章末題解已在前一步用頁碼排除，不會靠刪除「答案」等詞而誤刪教學內容。

不使用 LLM 改寫原文。清理後的文字會保存到知識庫，也會在搜尋後交給回答模型，讓依據可以核對。此函式是針對本份 PDF 的清理規則，不保證適用所有文件。

In [6]:
def extract_rag_text(raw_text):
    """移除本 PDF 的重複頁首與頁碼，保留可追溯的正文。"""
    text = raw_text.replace("\r\n", "\n").replace("\r", "\n").replace("\u00a0", " ")
    lines = []
    for raw_line in text.splitlines():
        line = re.sub(r"[ \t]+", " ", raw_line).strip()
        if not line:
            continue
        if re.fullmatch(r"[1-5]\s*-\s*\d+", line):
            continue
        if re.search(r"第[三四五]章", line) and "\uf07d" in line:
            continue
        # 這些私用區字元在本文件是項目符號或裝飾；不含正文文字。
        line = re.sub(r"[\ue000-\uf8ff]+", "•", line)
        line = re.sub(r"(?<=[\u4e00-\u9fff]) +(?=[\u4e00-\u9fff])", "", line)
        lines.append(line)
    cleaned = "\n".join(lines).strip()
    if not cleaned or "\ufffd" in cleaned:
        raise ValueError("正文為空或含無法解碼的替代字元，請核對原始 PDF。")
    return cleaned

sample_rag_text = extract_rag_text(sample_raw)
print(sample_rag_text[:1800])
print("清理後字元數：", len(sample_rag_text))

• 檢索增強生成（ RAG） ：這一方法將「外部知識檢索」 與「語言生成」結
合，能夠在生成過程中整合大量外部資料，提升模型在背景知識要求較高
任務中的準確性。
○ 示例 ： 在一個問答系統中 ，RAG 可以先檢索出相關的資料 （ 如從
Wikipedia 檢索某一領域的資料） ， 再基於這些資料生成對應的答案 。 例
如，在解答「 What are the symptoms of COVID -19?」時， RAG 系統首
先會從公共知識庫檢索相關資料 ， 再生成更具體的答案 。 或是在醫療摘
要任務中， RAG 可以從醫學文獻資料庫中檢索最新的研究資料，並將
其結合生成診斷或治療建議。
• 低秩自適應（ LoRA）：LoRA 是一種使機器學習模型能夠快速適應新環境
的技術 ， 透過向原始模型添加輕量級的參數 ， 而非直接修改整體模型結構 ，
來幫助模型在特定應用中進行快速調整。
這種方法使得大型和複雜的模型可以更靈活地適應不同的使用需求 ， 並能
在不大量重新訓練的情況下，高效地擴充到更多情境。 LoRA 為開發人員
提供了一種有效的方式 ， 使他們能夠快速地調整和擴充他們構建的機器學
習模型。
○ 示例：假設我們要在資源有限的邊緣設備上部署大型預訓練模型（如
GPT-3） ，我們可以利用LoRA 技術來微調模型，僅調整必要的低秩參
數，從而大幅減少計算和儲存需求，實現高效的部署。
隨著預訓練語言模型的進步，結合語境理解、生成能力、外部知識強化和高
效微調策略等技術，讓現代 NLP 模型在實務應用中能夠更加靈活和高效。透過
RAG 和 LoRA 等新技術的應用，未來的預訓練模型將在多工、多語言情境中發揮
更大的潛力，並且能夠快速適應不同語境和知識需求，提升各種自然語言處理任
務的表現。
清理後字元數： 757


## 6. 第二個修改函式：保存來源與頁碼

metadata 就像每段文字的資料卡。它記錄章節、原 PDF 網址、PDF 頁碼、書內頁碼和檔案指紋；搜尋後可用來顯示來源，不用請模型猜網址。

章節名稱來自這個版本的目錄與人工核對的起訖頁。檔名中的 `20251222101833` 只作版本辨識，不當成已確認的出版日期，所以 `date` 留空。每段的標題會進入 embedding；來源等其餘欄位則留作追溯使用。

In [7]:
def extract_metadata(raw_text, filename, pdf_page):
    """raw_text 保留老師的介面概念；本份主要靠已核對的頁碼表定位。"""
    matches = [row for row in SECTION_RANGES if row[0] <= pdf_page <= row[1]]
    if len(matches) != 1:
        raise ValueError(f"PDF 第 {pdf_page} 頁找不到唯一章節。")
    start, end, section, offset = matches[0]
    chapter = 3 if offset == 7 else 4 if offset == 94 else 5
    book_page = f"{chapter}-{pdf_page - offset}"
    return {
        "title": section,
        "section": section,
        "source_url": f"{SOURCE_URL}#page={pdf_page}",
        "source_pdf_url": SOURCE_URL,
        "pdf_page": pdf_page,
        "book_page": book_page,
        "filename": filename,
        "source_pdf_sha256": SOURCE_PDF_SHA256,
        "source_file_timestamp": "20251222101833",
        "date": None,
        "document_type": "官方學習指引正文文字擷取（非 OCR、非摘要）",
    }

sample_metadata = extract_metadata(sample_raw, SOURCE_FILENAME, SAMPLE_PDF_PAGE)
print(json.dumps(sample_metadata, ensure_ascii=False, indent=2))

{
  "title": "3.1 自然語言處理技術與應用",
  "section": "3.1 自然語言處理技術與應用",
  "source_url": "https://www.ipas.org.tw/api/proxy/uploads/certification_resource/bf93f438f7be48d295c1b40a34d79f3d/AI%E6%87%89%E7%94%A8%E8%A6%8F%E5%8A%83%E5%B8%AB%28%E4%B8%AD%E7%B4%9A%29-%E5%AD%B8%E7%BF%92%E6%8C%87%E5%BC%95-%E7%A7%91%E7%9B%AE1%E4%BA%BA%E5%B7%A5%E6%99%BA%E6%85%A7%E6%8A%80%E8%A1%93%E6%87%89%E7%94%A8%E8%A6%8F%E5%8A%83_20251222101833.pdf#page=36",
  "source_pdf_url": "https://www.ipas.org.tw/api/proxy/uploads/certification_resource/bf93f438f7be48d295c1b40a34d79f3d/AI%E6%87%89%E7%94%A8%E8%A6%8F%E5%8A%83%E5%B8%AB%28%E4%B8%AD%E7%B4%9A%29-%E5%AD%B8%E7%BF%92%E6%8C%87%E5%BC%95-%E7%A7%91%E7%9B%AE1%E4%BA%BA%E5%B7%A5%E6%99%BA%E6%85%A7%E6%8A%80%E8%A1%93%E6%87%89%E7%94%A8%E8%A6%8F%E5%8A%83_20251222101833.pdf",
  "pdf_page": 36,
  "book_page": "3-29",
  "filename": "ipas_ai_intermediate_subject1.pdf",
  "source_pdf_sha256": "7b1c67e3280600fbeca893bc1c13421d5aa44841272c34b456f1346731bf3812",
  "source_file_timestamp": "202

## 7. 頁內切段，再整理成老師使用的統一格式

把整本書做成一個向量，會太長且不容易精準找到重點。因此每頁清理後再切成最多 800 字元的小段，盡量在句號、分號或換行處結束；相鄰段重複 120 字元，補回一些邊界上下文。

每段得到 `doc_id`、`text` 和 `metadata`。`text` 加上章節標題，讓單獨讀到這一段時仍知道主題。`char_start`、`char_end` 記錄清理後頁內的位置，起點包含、終點不包含，方便重查。

**檢查重點：** 切段非空、每頁都有被保留、沒有漏掉字元，且沒有章末題解頁混進來。這是簡單且可說明的頁內切段；跨頁的段落仍可能被分開，後續可從 top-k 取回其他相關段落來補充。

In [8]:
def split_page_text(text, chunk_size=800, overlap=120):
    if not isinstance(chunk_size, int) or not isinstance(overlap, int) or not 0 <= overlap < chunk_size:
        raise ValueError("切段大小與重疊設定不合理。")
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            # 後 35% 範圍內有句尾，就優先在那裡結束；沒有則按字元切。
            lower = start + max(overlap + 1, int(chunk_size * 0.65))
            candidates = [text.rfind(mark, lower, end) for mark in ["。", "！", "？", "；", "\n"]]
            boundary = max(candidates)
            if boundary >= lower:
                end = boundary + 1
        if end <= start:
            raise ValueError("切段沒有向前移動。")
        if text[start:end].strip():
            chunks.append((start, end, text[start:end]))
        if end == len(text):
            break
        start = end - overlap
    return chunks

documents = []
page_summary = []
for pdf_page, raw_text in raw_pages.items():
    cleaned = extract_rag_text(raw_text)
    page_metadata = extract_metadata(raw_text, SOURCE_FILENAME, pdf_page)
    chunks = split_page_text(cleaned, CHUNK_SIZE, CHUNK_OVERLAP)
    coverage = np.zeros(len(cleaned), dtype=bool)
    for part, (start, end, body) in enumerate(chunks, start=1):
        coverage[start:end] = True
        metadata = {**page_metadata, "chunk_in_page": part, "char_start": start, "char_end": end}
        doc_id = f"doc_{len(documents) + 1:05d}"
        rag_text = f"{metadata['title']}\n\n{body}"
        documents.append({"doc_id": doc_id, "text": rag_text, "metadata": metadata})
    if not coverage.all():
        raise ValueError(f"PDF 第 {pdf_page} 頁有文字在切段時遺漏。")
    page_summary.append({"PDF頁碼": pdf_page, "書內頁碼": page_metadata["book_page"],
                         "章節": page_metadata["section"], "正文長度": len(cleaned), "切段數": len(chunks)})

index_doc_ids = [doc["doc_id"] for doc in documents]
assert documents and len(set(index_doc_ids)) == len(documents)
assert {d["metadata"]["pdf_page"] for d in documents} == set(selected_pages)
assert not ({d["metadata"]["pdf_page"] for d in documents} & excluded_pages)
assert not any(re.search(r"\bAns\s*[（(]", d["text"]) for d in documents)
summary = pd.DataFrame(page_summary)
display(summary.head(12))
display(summary.groupby("章節", sort=False).agg(頁數=("PDF頁碼", "count"), 切段數=("切段數", "sum")).reset_index())
print("正文頁數：", len(raw_pages), "／知識段落數：", len(documents))
print("第一段統一格式：")
print(json.dumps(documents[0], ensure_ascii=False, indent=2))

,PDF頁碼,書內頁碼,章節,正文長度,切段數
0,8,3-1,第三章 AI 相關技術應用｜概述,686,1
1,9,3-2,第三章 AI 相關技術應用｜概述,460,1
2,10,3-3,3.1 自然語言處理技術與應用,895,2
3,11,3-4,3.1 自然語言處理技術與應用,893,2
4,12,3-5,3.1 自然語言處理技術與應用,718,1
5,13,3-6,3.1 自然語言處理技術與應用,818,2
6,14,3-7,3.1 自然語言處理技術與應用,928,2
7,15,3-8,3.1 自然語言處理技術與應用,850,2
8,16,3-9,3.1 自然語言處理技術與應用,861,2
9,17,3-10,3.1 自然語言處理技術與應用,860,2


,章節,頁數,切段數
0,第三章 AI 相關技術應用｜概述,2,2
1,3.1 自然語言處理技術與應用,37,56
2,3.2 電腦視覺技術與應用,19,27
3,3.3 生成式 AI 技術與應用,14,19
4,3.4 多模態人工智慧應用,11,13
5,第四章 AI 導入評估規劃｜概述,2,3
6,4.1 AI 導入評估,12,13
7,4.2 AI 導入規劃,14,17
8,4.3 AI 風險管理,15,20
9,第五章 AI 技術應用與系統部署｜概述,1,1


正文頁數： 148 ／知識段落數： 198
第一段統一格式：
{
  "doc_id": "doc_00001",
  "text": "第三章 AI 相關技術應用｜概述\n\nAI\n面對當前 AI 技術的蓬勃發展，組織在導入人工智慧時常需要先理解各種核\n心技術的特點、應用範疇與成熟度，再與業務需求進行對應與結合。因此，本章\nAI 相關技術應用將從實務上常見的 AI 子領域切入，包括自然語言處理（ Natural\nLanguage Processing , NLP） 、電腦視覺（Computer Vision , CV） 、生成式AI，以及\n多模態 AI 等主題，說明它們各自在何種情境中能夠發揮關鍵價值。\n3.1 自然語言處理技術與應用\n自然語言處理是 AI 領域中最具挑戰性之一，旨在使機器能夠理解、解釋並生\n成人類語言。隨著語音助理、聊天機器人及智慧客服等應用的普及， NLP 技術的\n發展已經從簡單的語言分析進階到複雜的語意理解 。 此領域的技術涵蓋文本分類 、\n情感分析、命名實體辨識（ Named Entity Recognition , NER） 、機器翻譯等，廣泛應\n用於搜索引擎、情感分析、智慧客服、醫療診斷、法律文本分析等領域。 NLP 的\n核心價值在於能幫助企業處理大量文本資料，提取出有價值的資訊並進行智慧化\n處理。\n3.2 電腦視覺技術與應用\n電腦視覺技術使得機器能夠 “看懂”影像或影片 ，進而進行物體辨識 、場景理\n解及行為分析等工作。隨著智慧監控、無人駕駛及醫療影像診斷等應用的發展，\n電腦視覺技術在各行各業中扮演著越來越重要的角色。常見的應用情境包括人臉\n辨識 、物品檢測、視覺定位、視覺導航等。透過強大的圖像辨識能力，可以幫助\n企業自動化品質檢測、提升生產效率，並加速智慧城市建設與無人駕駛車輛的技\n術成熟。",
  "metadata": {
    "title": "第三章 AI 相關技術應用｜概述",
    "section": "第三章 AI 相關技術應用｜概述",
    "source_url": "https://www.ipas.org.tw/api/proxy/uploads/certification_resource/bf93f438f7be48d295c1b40a34d79f3d/A

## 8. 載入老師使用的 EmbeddingGemma 2

Embedding 可以想成替文字建立一組「語意座標」。相近主題的座標通常較接近，但不等於模型已經判斷內容正確。

先查詢模型目前確切的 commit，再鎖定這個版本載入，並將版本寫進知識庫。第二份才能使用同一套座標規則來理解問題。只啟用文字部分，不載入圖像或音訊編碼器。

首次下載需要等待；失敗時先確認網路、記憶體與套件版本。此處不會自動替換成另一個模型。

In [9]:
try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID, token=False).sha
    model = SentenceTransformer(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        device=DEVICE,
        config_kwargs={"vision_config": None, "audio_config": None},
        model_kwargs={"dtype": DTYPE, "attn_implementation": "eager"},
    )
except Exception as exc:
    raise RuntimeError(
        "EmbeddingGemma 2 載入失敗。請確認網路、套件版本，必要時重新啟動工作階段；"
        "若 GPU 記憶體不足，改用 CPU。未替換成其他模型。"
    ) from exc

print("模型：", MODEL_ID)
print("版本：", MODEL_REVISION)
print("裝置：", model.device)
print("模型精度：", model.dtype)
print("向量維度：", model.get_sentence_embedding_dimension())
print("模型單筆 token 上限：", model.max_seq_length)

模型： google/embeddinggemma-2
版本： f9972ac22e696a105c482202646979e33e4b5a1e
裝置： cpu
模型精度： torch.float32
向量維度： 768
模型單筆 token 上限： 1000000000000000019884624838656


## 9. 檢查每段真正送入模型的 token 長度

模型讀取的單位叫 token，不等於中文字數。文件使用老師的 `title: none | text: {text}` 前綴；因正文已含章節標題，所以不再把標題重複加進前綴。

這格會檢查「包含前綴與標題」的實際長度，最多採 2048 tokens 的教學預算。超過就停止，避免模型只讀前半段、我們卻以為全文都有被搜尋。若超過，調小切段大小，再從第 7 步重建。

In [10]:
document_inputs = [DOCUMENT_TEMPLATE.format(text=doc["text"]) for doc in documents]
reported_limit = int(model.max_seq_length)
TOKEN_LIMIT = min(2048, reported_limit)
if TOKEN_LIMIT <= 0:
    raise ValueError("模型沒有有效的輸入長度上限。")
model.max_seq_length = TOKEN_LIMIT

def count_input_tokens(text):
    return len(model.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])

token_counts = [count_input_tokens(text) for text in document_inputs]
display(pd.DataFrame({"doc_id": index_doc_ids, "tokens_含前綴": token_counts}).describe(include="all"))
long_docs = [doc_id for doc_id, length in zip(index_doc_ids, token_counts) if length > TOKEN_LIMIT]
if long_docs:
    raise ValueError(f"以下段落超過 {TOKEN_LIMIT} tokens，請縮小 CHUNK_SIZE 後重建：{long_docs}")
print("token 上限：", TOKEN_LIMIT, "／本次最長段落：", max(token_counts))
print("第一段送入模型的文字（節錄）：\n", document_inputs[0][:600])

,doc_id,tokens_含前綴
count,198,198.000000
unique,198,NaN
top,doc_00001,NaN
freq,1,NaN
mean,NaN,375.035354
std,NaN,149.838742
min,NaN,85.000000
25%,NaN,199.750000
50%,NaN,449.500000
75%,NaN,485.000000


token 上限： 2048 ／本次最長段落： 558
第一段送入模型的文字（節錄）：
 title: none | text: 第三章 AI 相關技術應用｜概述

AI
面對當前 AI 技術的蓬勃發展，組織在導入人工智慧時常需要先理解各種核
心技術的特點、應用範疇與成熟度，再與業務需求進行對應與結合。因此，本章
AI 相關技術應用將從實務上常見的 AI 子領域切入，包括自然語言處理（ Natural
Language Processing , NLP） 、電腦視覺（Computer Vision , CV） 、生成式AI，以及
多模態 AI 等主題，說明它們各自在何種情境中能夠發揮關鍵價值。
3.1 自然語言處理技術與應用
自然語言處理是 AI 領域中最具挑戰性之一，旨在使機器能夠理解、解釋並生
成人類語言。隨著語音助理、聊天機器人及智慧客服等應用的普及， NLP 技術的
發展已經從簡單的語言分析進階到複雜的語意理解 。 此領域的技術涵蓋文本分類 、
情感分析、命名實體辨識（ Named Entity Recognition , NER） 、機器翻譯等，廣泛應
用於搜索引擎、情感分析、智慧客服、醫療診斷、法律文本分析等領域。 NLP 的
核心價值在於能幫助企業處理大量文本資料，提取出有價值的資訊並進行智慧化
處理。
3.2 電腦視覺技術與應用
電腦視覺技術使得機器能夠 “看懂”影像或影片 ，進而進行物體辨識 、場景理
解及行為分析等工作。隨著智慧監控、無人駕駛及醫療影


## 10. 產生向量：把每個段落轉成可比較的數字

一段對應向量矩陣的一列，欄數是模型向量維度。`prompt=""` 代表前綴已自己加入，避免模型再加一次。正規化會把每列向量的長度調成約 1，後面的內積就能用來計算 cosine similarity。

程式檢查列數、維度、有限值與零向量，最後轉成 FAISS 使用的 float32 格式。產生向量是在使用既有模型，沒有訓練或微調模型。

In [11]:
embeddings = model.encode(
    document_inputs,
    prompt="",
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
if embeddings.ndim != 2 or embeddings.shape[0] != len(documents):
    raise ValueError("向量形狀與文件數不符。")
if not np.isfinite(embeddings).all():
    raise ValueError("向量含有 NaN 或無限值。")
if np.any(np.linalg.norm(embeddings, axis=1) <= 0):
    raise ValueError("出現零向量。")
faiss.normalize_L2(embeddings)
DIMENSION = int(embeddings.shape[1])
if DIMENSION != int(model.get_sentence_embedding_dimension()):
    raise ValueError("實際向量維度與模型宣告不符。")
print("矩陣形狀（文件數, 向量維度）：", embeddings.shape)
print("第一段向量的前10個數字：", embeddings[0, :10])
print("前5段向量長度：", np.linalg.norm(embeddings[:5], axis=1))

矩陣形狀（文件數, 向量維度）： (198, 768)
第一段向量的前10個數字： [ 0.00985135 -0.0020542   0.00058328  0.03876187  0.06419474 -0.01256225
 -0.00631477  0.01748265  0.02561953 -0.08557297]
前5段向量長度： [1. 1. 1. 1. 1.]


## 11. 建立 FAISS 精確搜尋索引

`IndexFlatIP` 比較問題向量與所有文件向量的內積。因為雙方都正規化，排名就是 cosine similarity 的排名。

FAISS 回傳的是向量編號，**不是原文**。`index_doc_ids` 保存向量順序與文件編號的對應，再透過 `doc_id` 找回正文與 metadata。

In [12]:
index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)
documents_by_id = {doc["doc_id"]: doc for doc in documents}
assert index.ntotal == len(index_doc_ids) == len(documents)
print("索引類型：", type(index).__name__)
print("索引向量數：", index.ntotal)
print("索引維度：", index.d)
display(pd.DataFrame({"FAISS編號": range(len(index_doc_ids)), "doc_id": index_doc_ids}))

索引類型： IndexFlatIP
索引向量數： 198
索引維度： 768


,FAISS編號,doc_id
0,0,doc_00001
1,1,doc_00002
2,2,doc_00003
3,3,doc_00004
4,4,doc_00005
...,...,...
193,193,doc_00194
194,194,doc_00195
195,195,doc_00196
196,196,doc_00197


## 12. 定義問題向量化與搜尋函式

問題也要用同一個 embedding 模型向量化，但使用查詢專用前綴。`top_k` 表示取回幾段文字；程式限制它不超過實際文件數。

搜尋函式只做檢索，還沒有呼叫 LLM。精確搜尋保證依目前向量找到分數最高者，不保證文件一定足以回答問題。

In [13]:
def embed_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    if count_input_tokens(query_input) > TOKEN_LIMIT:
        raise ValueError("問題超過模型輸入長度上限，請縮短問題。")
    query_vector = model.encode(
        [query_input], prompt="", convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, DIMENSION) or not np.isfinite(query_vector).all():
        raise ValueError("問題向量的維度或數值不正確。")
    if np.linalg.norm(query_vector) <= 0:
        raise ValueError("問題產生零向量。")
    faiss.normalize_L2(query_vector)
    return query_vector


def search_documents(question, top_k=3):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k 必須是大於0的整數。")
    query_vector = embed_question(question)
    scores, positions = index.search(query_vector, min(int(top_k), index.ntotal))
    results = []
    for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1):
        doc_id = index_doc_ids[int(position)]
        results.append({"rank": rank, "score": float(score), **documents_by_id[doc_id]})
    return results

## 13. 試查一次：先確認找到的證據是否相關

搜尋先把問題轉成向量，再用 FAISS 找最相近的段落，最後依向量位置取回原文與 metadata。到這裡還沒有 LLM 生成答案。

預設問題是「RAG 如何結合外部知識檢索與語言生成？」請打開取回段落的來源頁碼核對。可以比較 `top_k=1`、`3`、`5`：多取幾段可能補足資訊，也可能增加不相關內容。

也可以問「我這次 iPAS 考試會得幾分？」觀察即使沒有答案，FAISS 仍會回傳相近段落。相似度只是排序依據，不是答案正確率。

In [14]:
QUESTION = "RAG 如何結合外部知識檢索與語言生成？"
TOP_K = 3
results = search_documents(QUESTION, TOP_K)
display(pd.DataFrame([
    {"排名": r["rank"], "doc_id": r["doc_id"], "章節": r["metadata"]["section"],
     "PDF頁碼": r["metadata"]["pdf_page"], "書內頁碼": r["metadata"]["book_page"],
     "相似度": round(r["score"], 4)}
    for r in results
]))
for result in results:
    print(f"\n[{result['doc_id']}] PDF 第 {result['metadata']['pdf_page']} 頁／書內 {result['metadata']['book_page']}")
    print(result["text"])
    print("來源：", result["metadata"]["source_url"])

,排名,doc_id,章節,PDF頁碼,書內頁碼,相似度
0,1,doc_00047,3.1 自然語言處理技術與應用,36,3-29,0.8075
1,2,doc_00046,3.1 自然語言處理技術與應用,35,3-28,0.7318
2,3,doc_00042,3.1 自然語言處理技術與應用,33,3-26,0.7092



[doc_00047] PDF 第 36 頁／書內 3-29
3.1 自然語言處理技術與應用

• 檢索增強生成（ RAG） ：這一方法將「外部知識檢索」 與「語言生成」結
合，能夠在生成過程中整合大量外部資料，提升模型在背景知識要求較高
任務中的準確性。
○ 示例 ： 在一個問答系統中 ，RAG 可以先檢索出相關的資料 （ 如從
Wikipedia 檢索某一領域的資料） ， 再基於這些資料生成對應的答案 。 例
如，在解答「 What are the symptoms of COVID -19?」時， RAG 系統首
先會從公共知識庫檢索相關資料 ， 再生成更具體的答案 。 或是在醫療摘
要任務中， RAG 可以從醫學文獻資料庫中檢索最新的研究資料，並將
其結合生成診斷或治療建議。
• 低秩自適應（ LoRA）：LoRA 是一種使機器學習模型能夠快速適應新環境
的技術 ， 透過向原始模型添加輕量級的參數 ， 而非直接修改整體模型結構 ，
來幫助模型在特定應用中進行快速調整。
這種方法使得大型和複雜的模型可以更靈活地適應不同的使用需求 ， 並能
在不大量重新訓練的情況下，高效地擴充到更多情境。 LoRA 為開發人員
提供了一種有效的方式 ， 使他們能夠快速地調整和擴充他們構建的機器學
習模型。
○ 示例：假設我們要在資源有限的邊緣設備上部署大型預訓練模型（如
GPT-3） ，我們可以利用LoRA 技術來微調模型，僅調整必要的低秩參
數，從而大幅減少計算和儲存需求，實現高效的部署。
隨著預訓練語言模型的進步，結合語境理解、生成能力、外部知識強化和高
效微調策略等技術，讓現代 NLP 模型在實務應用中能夠更加靈活和高效。透過
RAG 和 LoRA 等新技術的應用，未來的預訓練模型將在多工、多語言情境中發揮
更大的潛力，並且能夠快速適應不同語境和知識需求，提升各種自然語言處理任
務的表現。
來源： https://www.ipas.org.tw/api/proxy/uploads/certification_resource/bf93f438f7be48d295c1b40a34d79f3d/AI%E6%87%89%E7%94%A8%E8%A6%8F%E5%8A%83%E5%B8%AB%28%E4%B8%AD%E7%B4%9A%29-%E5%AD%B8%E7%BF%92%

## 14. 保存索引、文件、編號對照與設定

第二份 Colab 需要下面四個檔案，不能只拿 FAISS 索引：

| 檔案 | 用途 |
|---|---|
| `index.faiss` | 文件向量與搜尋索引 |
| `documents.json` | 取回後提供給 LLM 的正文與來源資訊 |
| `index_doc_ids.json` | FAISS 編號對應的文件編號 |
| `config.json` | 模型確切版本、前綴、維度、正規化、套件版本及檢查資訊 |

不保存模型權重、API 金鑰或其他環境變數。第二份 Colab 載入相同模型版本，只重新計算問題向量。

In [15]:
EXPORT_DIR = Path(tempfile.mkdtemp(prefix="rag_index_"))
faiss.write_index(index, str(EXPORT_DIR / "index.faiss"))

for filename, data in [("documents.json", documents), ("index_doc_ids.json", index_doc_ids)]:
    (EXPORT_DIR / filename).write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
    )

config = {
    "format_version": "1.0",
    "created_at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat(timespec="seconds"),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_config_kwargs": {"vision_config": None, "audio_config": None},
    "dimension": DIMENSION,
    "normalize_embeddings": True,
    "index_type": "IndexFlatIP",
    "metric": "cosine_similarity_via_normalized_inner_product",
    "document_template": DOCUMENT_TEMPLATE,
    "query_template": QUERY_TEMPLATE,
    "encode_prompt": "",
    "max_seq_length": TOKEN_LIMIT,
    "document_count": len(documents),
    "document_unit": "pdf_page_chunk",
    "document_text_includes_title": True,
    "package_versions": PACKAGE_VERSIONS,
    "build_device": DEVICE,
    "build_dtype": str(DTYPE),
    "project_id": "ipas-ai-intermediate-subject1",
    "source_pdf_url": SOURCE_URL,
    "source_pdf_sha256": SOURCE_PDF_SHA256,
    "source_pdf_total_pages": len(reader.pages),
    "content_page_ranges": CONTENT_RANGES,
    "content_page_count": len(raw_pages),
    "section_ranges": SECTION_RANGES,
    "chunking": {"method": "page_local_character_with_boundary", "chunk_size": CHUNK_SIZE, "overlap": CHUNK_OVERLAP},
    "file_sha256": {
        name: hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
        for name in ["index.faiss", "documents.json", "index_doc_ids.json"]
    },
}
(EXPORT_DIR / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("已保存：")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")
print("\n模型版本及前綴已寫入 config.json。")

已保存：
config.json (3,145 bytes)
documents.json (552,413 bytes)
index.faiss (608,301 bytes)
index_doc_ids.json (2,972 bytes)

模型版本及前綴已寫入 config.json。


## 15. 重新載入，驗證第二份 Colab 所需的資料

從剛才保存的檔案重新載入，檢查完整性、文件編號與向量對應，再使用**同一個問題向量**比對保存前後的排名與分數。此處不重算問題向量，以免不同運算裝置的細微浮點誤差干擾索引保存檢查。

也將 FAISS 結果與 NumPy 的全矩陣內積排序比較，確認精確搜尋的意義。遇到完全同分時，兩套工具的同分排列順序可以不同。

In [16]:
loaded_config = json.loads((EXPORT_DIR / "config.json").read_text(encoding="utf-8"))
for name, expected_hash in loaded_config["file_sha256"].items():
    actual_hash = hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"檔案完整性檢查失敗：{name}"

loaded_index = faiss.read_index(str(EXPORT_DIR / "index.faiss"))
loaded_documents = json.loads((EXPORT_DIR / "documents.json").read_text(encoding="utf-8"))
loaded_ids = json.loads((EXPORT_DIR / "index_doc_ids.json").read_text(encoding="utf-8"))
assert loaded_index.ntotal == len(loaded_documents) == len(loaded_ids) == loaded_config["document_count"]
assert loaded_index.d == loaded_config["dimension"]
assert len(set(loaded_ids)) == len(loaded_ids)
assert set(loaded_ids) == {doc["doc_id"] for doc in loaded_documents}
assert loaded_documents == documents and loaded_ids == index_doc_ids

verification_query = embed_question(QUESTION)
k = min(TOP_K, index.ntotal)
before_scores, before_positions = index.search(verification_query, k)
after_scores, after_positions = loaded_index.search(verification_query, k)
np.testing.assert_array_equal(before_positions, after_positions)
np.testing.assert_allclose(before_scores, after_scores, atol=1e-6, rtol=1e-6)

all_scores = (embeddings @ verification_query[0]).ravel()
expected_top_scores = np.sort(all_scores)[::-1][:k]
np.testing.assert_allclose(after_scores[0], expected_top_scores, atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(after_scores[0], all_scores[after_positions[0]], atol=1e-5, rtol=1e-5)
print("通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。")

通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。


## 16. 打包並下載，記下 ZIP 的指紋

ZIP 根目錄包含老師定義的四個檔案。不能只上傳 `index.faiss`：搜尋後還需要正文、編號對照與模型設定。模型權重、原始 PDF 和 API 金鑰不放入 ZIP。

下載名稱固定為 `ipas_vector_db.zip`，以後請將它放在 GitHub 的 `knowledge_base/` 資料夾。下方會印出 ZIP 的 SHA-256，將它填進 RAG02 的 `EXPECTED_ZIP_SHA256`；重新建庫的時間或內容不同，都可能改變指紋，要一起更新。

In [17]:
EXPORT_ZIP = Path.cwd() / EXPORT_ZIP_NAME
required_files = ["index.faiss", "documents.json", "index_doc_ids.json", "config.json"]
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in required_files:
        archive.write(EXPORT_DIR / name, arcname=name)
with zipfile.ZipFile(EXPORT_ZIP) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)
ZIP_SHA256 = hashlib.sha256(EXPORT_ZIP.read_bytes()).hexdigest()
print("下載檔名：", EXPORT_ZIP_NAME)
print("大小：", f"{EXPORT_ZIP.stat().st_size:,} bytes")
print("請複製到 RAG02：")
print(f'EXPECTED_ZIP_SHA256 = "{ZIP_SHA256}"')
try:
    from google.colab import files
except ImportError:
    print("本機輸出位置：", EXPORT_ZIP.resolve())
else:
    files.download(str(EXPORT_ZIP))

下載檔名： ipas_vector_db.zip
大小： 691,192 bytes
請複製到 RAG02：
EXPECTED_ZIP_SHA256 = "aad48241d12078a42f1ff1b3fe526d9211e3f85f088cdc005f2003619778b45c"


## 17. 手動上傳 GitHub，讓 RAG02 能獨立使用

1. 登入 GitHub，建立 repository，例如 `ipas-rag-homework`。若要讓老師不用登入就能下載，可使用公開 repository；先確認教材擷取內容的分享範圍符合你的使用情境。
2. 在 repository 選 **Add file → Create new file**，輸入 `knowledge_base/.gitkeep`，儲存後就會有資料夾。
3. 進入 `knowledge_base`，選 **Add file → Upload files**，上傳剛下載的 **`ipas_vector_db.zip`**，再按 **Commit changes**。ZIP 不需要解壓縮。
4. 回到 repository 根目錄，上傳 RAG01、RAG02 及 README。不要上傳 API 金鑰、`.env` 或模型權重。
5. 在 RAG02 設定 `GITHUB_OWNER` 為帳號、`GITHUB_REPO` 為 repository 名稱、`GITHUB_REF` 為分支或 commit；檔案路徑使用 `knowledge_base/ipas_vector_db.zip`。
6. 將上一步的 ZIP SHA-256 複製到 RAG02。先完成上傳與設定，再在**新的 Colab 工作階段**由上往下執行 RAG02。

GitHub 網址中的 `blob` 頁面是網頁，不是 ZIP 本身；RAG02 會自己組成 `raw.githubusercontent.com` 的檔案下載網址。預設分支是 `main`，若你的分支不同要修改。想固定交作業的版本，可把 `GITHUB_REF` 改成該次提交的完整 commit SHA。

**這份 Notebook 不會自動建立或推送 repository。** 你也可以先用 RAG02 的 `upload` 模式測試同一份 ZIP，確認流程後再切回 GitHub 模式；最終交作業應確認 GitHub 模式可下載。

### 與老師範例的對照

| 老師提醒的項目 | 這份作業的做法 |
|---|---|
| 修改 `extract_rag_text()` | 改讀 PDF 頁面，清除頁首與頁碼 |
| 修改 `extract_metadata()` | 記錄章節、兩套頁碼、原始網址與 PDF 指紋 |
| 不把練習問題與答案放入正文 | 用已核對的題解頁碼排除 |
| 長文不能默默截斷 | 頁內切段後，檢查包含前綴的 token 數 |
| 同一模型與查詢前綴 | 保存模型 commit，RAG02 原樣載入 |
| 保存後要重新載入驗證 | 核對雜湊、向量位置、排名與分數 |
| 相似度不等於正確率 | 在搜尋練習與 Gradio 介面提醒 |

### 作業觀察，請執行後自行填寫

- 比較同一題的 top-k=1、3、5：取回哪些頁？哪些段落支持答案？
- 試問一題教材沒有的資訊：模型是否承認不足？
- 找一個切段邊界或跨頁段落：重疊有什麼幫助，又有哪些限制？
- 修改切段大小後，段落數與檢索結果如何改變？沒有做實驗時，不要寫成已完成的結果。

### 參考資料

- [iPAS 官方學習指引（本次來源）](https://www.ipas.org.tw/api/proxy/uploads/certification_resource/bf93f438f7be48d295c1b40a34d79f3d/AI%E6%87%89%E7%94%A8%E8%A6%8F%E5%8A%83%E5%B8%AB%28%E4%B8%AD%E7%B4%9A%29-%E5%AD%B8%E7%BF%92%E6%8C%87%E5%BC%95-%E7%A7%91%E7%9B%AE1%E4%BA%BA%E5%B7%A5%E6%99%BA%E6%85%A7%E6%8A%80%E8%A1%93%E6%87%89%E7%94%A8%E8%A6%8F%E5%8A%83_20251222101833.pdf)
- [EmbeddingGemma 2 官方 Sentence Transformers 指引](https://ai.google.dev/gemma/docs/embeddinggemma/inference-embeddinggemma-with-sentence-transformers)
- [FAISS 入門](https://github.com/facebookresearch/faiss/wiki/Getting-started)
- [pypdf 文字擷取限制](https://pypdf.readthedocs.io/en/stable/user/extract-text.html)